# Fine tune the model with Rosetta Code pairs

Creates an adapter using Rosetta Code pairs (see `MODEL.md`). Uses `train.jsonl` and `validation.jsonl`, which should be generated using `rosetta_write.py` and pushed to a branch.

Monitor the training and validation loss as fine tuning proceeds. With 3,000 - 5,000 Rosetta Code pairs, overfitting may appear a few checkpoints in, signaled by a sudden drop in the training loss and a gain in the validation loss. If this occurs, no more training is needed.

Runs can take 5-6 hours for 5,000 pairs, which may result in GPU exhaustion.

Before running: **Runtime → Change runtime type → T4 GPU.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SEED = 3407

In [ ]:
import os, subprocess
REPO = "https://github.com/05ryt31/CS152-Project.git"
if not os.path.isdir("CS152-Project"):
    subprocess.run(["git", "clone", "-q", "-b", "feat/finetuning", REPO], check=True)
os.chdir("CS152-Project")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/CS152/adapter/checkpoints"
import os;
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
%pip install -q "transformers>=4.45,<5" unsloth[colab-new] trl peft bitsandbytes datasets accelerate

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "pipeline/data/train.jsonl",
        "validation": "pipeline/data/validation.jsonl",
    },
)

In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
max_seq_length = 2048

dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=dtype,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    max_length=max_seq_length,
    completion_only_loss=True,
    packing=False,

    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),

    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

trainer.train()

In [ ]:
# In case of runtime disconnection, resume from the latest checkpoint by running this cell

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

print("Last checkpoint:", last_checkpoint)

if last_checkpoint is not None:
    print("Resuming from checkpoint")
    trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("Checkpoint not found; starting training from beginning")
    trainer.train()

In [ ]:
# To use a specific checkpoint for the adapter, run this cell

from unsloth import FastLanguageModel
from peft import PeftModel

CHECKPOINT = (CHECKPOINT_DIR / "checkpoint-xxx")

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=dtype,
)

model = PeftModel.from_pretrained(
    model,
    CHECKPOINT,
    is_trainable=False,
)

In [ ]:
# Run to save the adapter locally

ADAPTER_DIR = "/content/drive/MyDrive/CS152/adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)